<div dir="rtl">

# يقين: التشغيل والاختبار في كولاب

**كيف تستخدمين هذا الدفتر:** كل مربع رمادي فيه كود. مرّري الفأرة عليه واضغطي زر التشغيل ▶ على يساره، وانتظري حتى تظهر علامة ✓ خضراء، ثم انتقلي للمربع الذي بعده. شغّليها بالترتيب من الأعلى للأسفل.

**قبل البدء (مرة واحدة):** من القائمة العلوية **Runtime ← Change runtime type** واختاري **T4 GPU** ثم **Save**. هذا يجعل الخطوة 5 أسرع بكثير.

إذا ظهرت رسالة حمراء في أي خطوة، انسخيها وأرسليها لي.
</div>

<div dir="rtl">

## 1. إحضار الكود
ينسخ مستودع يقين إلى كولاب.
</div>

In [ ]:
%cd /content
!rm -rf Yaqeen
!git clone -q -b extension-mv3 https://github.com/sabascodes/Yaqeen.git
%cd /content/Yaqeen/backend
print("✓ تم إحضار الكود")

<div dir="rtl">

## 2. تثبيت المكتبات
تأخذ دقيقة أو دقيقتين.
</div>

In [ ]:
!pip install -q -r requirements-dev.txt -r requirements-semantic.txt
print("✓ تم التثبيت")

<div dir="rtl">

## 3. اختبارات خدمة Python
في آخر سطر يجب أن تري `passed` بدون `failed`.
</div>

In [ ]:
!python -m pytest -q

<div dir="rtl">

## 4. تنزيل نص المصحف من موسوعة القرآن الكريم
ينزّل 114 سورة. في النهاية يجب أن تري: حُفظت 6236 آية.
</div>

In [ ]:
!python fetch_quran.py

<div dir="rtl">

## 5. تنزيل أحاديث موسوعة الأحاديث النبوية وإنشاء تمثيلاتها
أطول خطوة: ينزّل الأحاديث واحدًا واحدًا ثم يحوّلها بالنموذج multilingual-e5-base. اتركيها حتى تنتهي.
</div>

In [ ]:
!python build_hadith.py

<div dir="rtl">

## 6. جرّبي الفحص بنفسك
اكتبي أي نص في الخانة `النص` على اليمين، ثم شغّلي المربع. يمكنك تغيير النص وتشغيله مرة بعد مرة.
أول تشغيل يحمّل النموذج فيأخذ وقتًا أطول.
</div>

In [ ]:
النص = "قال رسول الله ﷺ: إنما الأعمال بالنيات وإنما لكل امرئ ما نوى"  #@param {type:"string"}

import os, sys
os.chdir("/content/Yaqeen/backend"); sys.path.insert(0, ".")
from fastapi.testclient import TestClient
import main
r = TestClient(main.app).post("/verify", json={"text": النص}).json()
if r.get("status") == "error":
    print("تعذر الوصول إلى الدرر السنية الآن، جرّبي مرة أخرى.")
card = r.get("card", r)
for k, v in card.items():
    if v:
        print(f"{k}: {v}")

<div dir="rtl">

للتأكد أن كل شيء مربوط: شغّلي المربع التالي. يجب أن تري `quran_ayat: 6236`، وعددًا أكبر من صفر في `hadeethenc_hadith`، و`True` في `semantic` و`hadith_semantic`.
</div>

In [ ]:
from fastapi.testclient import TestClient
import main
for k, v in TestClient(main.app).get("/").json().items():
    print(f"{k}: {v}")

<div dir="rtl">

## 7. بناء إضافة المتصفح
يثبّت Node.js ثم يختبر الإضافة ويبنيها، ويضمّن فيها تمثيلات القرآن والأحاديث. في النهاية يُنزَّل ملف `yaqeen-extension.zip` إلى جهازك.
</div>

In [ ]:
%cd /content/Yaqeen/extension
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1
!apt-get install -y -qq nodejs > /dev/null 2>&1
!node -v
!npm install --ignore-scripts --silent
!npm test
!npm run build
!cd dist && zip -qr /content/yaqeen-extension.zip .
from google.colab import files
files.download("/content/yaqeen-extension.zip")

<div dir="rtl">

## 8. (اختياري) حفظ ملفات البيانات
كولاب يمسح كل شيء عند إغلاقه. هذا يحفظ لك ملفات الخطوتين 4 و5 حتى لا تعيديها، ويمكنك إرسال ملف التمثيلات لي لأضيفه إلى المستودع.
</div>

In [ ]:
%cd /content/Yaqeen/backend
!zip -qj /content/yaqeen-data.zip data/quran.json data/hadeethenc.json data/hadith_embeddings.npy data/hadith_embeddings_ids.json
from google.colab import files
files.download("/content/yaqeen-data.zip")

<div dir="rtl">

## 9. تركيب الإضافة في Chrome أو Edge (على جهازك، ليس في كولاب)
1. افتحي ملف `yaqeen-extension.zip` الذي نزل، واستخرجي ما فيه إلى مجلد (اضغطي عليه بالزر الأيمن ← Extract All).
2. في Chrome اكتبي في شريط العنوان: `chrome://extensions` (وفي Edge: `edge://extensions`).
3. فعّلي **Developer mode** (وضع المطوّر) من الزاوية.
4. اضغطي **Load unpacked** واختاري المجلد الذي استخرجتِه.
5. ستفتح صفحة إعدادات يقين: اضغطي **تنزيل البيانات الآن** وانتظري حتى يظهر عدد الآيات (6236) وعدد الأحاديث.
6. اضغطي أيقونة يقين بجانب شريط العنوان، والصقي آية أو حديثًا، واضغطي فحص.
7. افتحي X أو Facebook أو TikTok وسترين علامة يقين بجانب المنشورات التي فيها آيات أو أحاديث.
</div>